# MIG Manager

A practical reference for the **NVIDIA MIG Manager** — the tooling that
configures **Multi-Instance GPU (MIG)** partitions declaratively, both on a
single host (`nvidia-mig-parted`) and across a Kubernetes cluster
(`k8s-mig-manager`, the `nvidia.com/mig.config` controller shipped with the
GPU Operator).

MIG itself slices an A100/H100/A30/B200-class GPU into up to 7 isolated
instances, each with dedicated SMs, L2 cache slices, and memory. MIG Manager is
how you go from "MIG-capable hardware" to "the exact partition layout my
workloads need" — safely, repeatably, and without hand-running `nvidia-smi mig`
on every node.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

"MIG Manager" refers to two closely related pieces of NVIDIA tooling:

- **`nvidia-mig-parted`** (the *MIG Partition Editor*, repo `NVIDIA/mig-parted`):
  a host-level CLI that applies a **declarative YAML** describing the MIG layout
  you want. Instead of issuing a sequence of imperative `nvidia-smi mig -cgi /
  -cci` commands, you name a configuration (e.g. `all-1g.10gb`) and run
  `nvidia-mig-parted apply`. It reconciles the GPU to that target state.

- **`k8s-mig-manager`** (the *Kubernetes MIG Manager*, repo
  `NVIDIA/mig-parted/deployments/gpu-operator`): a **DaemonSet** that runs on GPU
  nodes and watches the `nvidia.com/mig.config` node label. When the label
  changes, it cordons/drains the node, stops the GPU operands, calls
  `nvidia-mig-parted` to apply the named config from a ConfigMap, then restores
  the node. It is installed and driven by the **NVIDIA GPU Operator**.

### Why use it?

- **Declarative, idempotent MIG config** — describe the end state once; applying
  the same config twice is a no-op. No fragile ordering of create/destroy
  commands.
- **Fleet-wide consistency** — label a node (or many) with a config name and the
  k8s manager rolls it out, handling the disruptive parts (draining clients,
  resetting the GPU) for you.
- **Safe reconfiguration** — MIG mode changes and instance create/destroy require
  that **no CUDA clients hold the GPU**. The manager orchestrates draining and,
  where required, the GPU reset/reboot so you don't corrupt running work.
- **Integrates with the rest of the stack** — after applying, it triggers the
  device plugin and GPU Feature Discovery to re-advertise the new
  `nvidia.com/mig-*` resources to the scheduler.

### When to use it?

- You run **A100 / A30 / H100 / H200 / B200** (or other MIG-capable) GPUs and
  want to **right-size** instances to many small/medium jobs (inference, notebooks,
  CI) rather than handing each job a whole GPU.
- You need **hard isolation** (fault, memory-bandwidth, and error isolation)
  between tenants on the same physical GPU — stronger than time-slicing or MPS.
- You manage GPUs **at scale on Kubernetes** and want partition layout to be part
  of declarative cluster config, not a manual SSH chore.

## Key Features

### Core capabilities of MIG Manager

| Capability | Description | Why it matters |
|---|---|---|
| Declarative configs | A `version: v1` YAML maps config *names* to per-device MIG layouts | Reproducible, reviewable, idempotent partitioning |
| `apply` / `export` / `assert` | Reconcile to a config, dump current layout as YAML, or verify state matches | Fits CI/GitOps: assert in checks, apply on rollout |
| Mode + geometry handling | Enables/disables MIG mode and creates GPU **instances (GI)** and **compute instances (CI)** | One command goes from MIG-off to a full partition set |
| Node-label driven (k8s) | `nvidia.com/mig.config` selects a named config per node | Change a label → cluster reconfigures that node |
| Drain & restore orchestration | Cordons node, evicts GPU pods, stops/starts operands, reboots if needed | Reconfig without corrupting running CUDA clients |
| State reporting | `nvidia.com/mig.config.state` = `pending` / `success` / `failed` | Observable rollout; alert on `failed` |
| MIG strategy support | Works with device-plugin/GFD `single` vs `mixed` strategies | Controls how MIG devices appear as schedulable resources |
| Hooks | `nvidia-mig-parted apply --hooks-file` runs pre/post steps | Stop services, reset GPU, restart operands cleanly |

## Architecture Overview

```text
                         Kubernetes control plane
                 (you set  nvidia.com/mig.config=<name>  on a node)
                                     |
   +--------------------------------- GPU node ---------------------------------+
   |                                                                            |
   |   k8s-mig-manager (DaemonSet)                                              |
   |     1. sees label change -> sets mig.config.state=pending                 |
   |     2. cordon + drain GPU pods                                            |
   |     3. stop operands: device-plugin, gpu-feature-discovery, dcgm-exporter |
   |     4. call  nvidia-mig-parted apply -c <name> -f /config/config.yaml      |
   |              (reads named config from the mig-parted ConfigMap)            |
   |             |                                                              |
   |             v                                                              |
   |   nvidia-mig-parted  --hooks-file hooks.yaml                               |
   |     - toggle MIG mode (may require GPU reset / node reboot)                |
   |     - create/destroy GPU Instances (GI) and Compute Instances (CI)         |
   |             |                                                              |
   |             v                                                              |
   |   NVIDIA driver / NVML  ->  hardware MIG partitions                        |
   |     5. restart operands -> device-plugin & GFD re-advertise resources       |
   |     6. uncordon, set mig.config.state=success                              |
   +----------------------------------------------------------------------------+
                                     |
                  scheduler now sees  nvidia.com/mig-1g.10gb: N  (mixed)
                            or        nvidia.com/gpu: N           (single)
```

### Components

1. **`nvidia-mig-parted` (mig-parted)**: the host CLI and library that actually
   talks to NVML to enable MIG mode and build the GI/CI geometry. Everything else
   is orchestration around it.
2. **MIG config (`config.yaml`)**: the declarative source of truth — a set of
   named configurations. In k8s it lives in a ConfigMap (default
   `default-mig-parted-config`).
3. **Hooks file (`hooks.yaml`)**: ordered shell steps `mig-parted` runs at
   defined points (`pre-apply-mode`, `pre-apply-config`, etc.) to stop GPU
   clients, reset the GPU, and restart them.
4. **`k8s-mig-manager` (DaemonSet)**: the controller that watches node labels,
   drains, invokes `mig-parted` + hooks, and reports state.
5. **GPU Operator**: installs and configures all of the above, wiring the
   manager to the device plugin, GFD, and DCGM exporter.

## Installation

### Prerequisites

- A **MIG-capable GPU**: A100, A30, H100, H200, GH200, B200, etc. (consumer and
  V100/T4-class cards do **not** support MIG).
- **NVIDIA driver** installed (or managed by the GPU Operator). MIG support has
  been in the data-center driver since R450.
- For the host CLI: download the `nvidia-mig-parted` binary (it is also baked
  into the GPU Operator's MIG-manager image).
- For Kubernetes: a cluster with the **NVIDIA GPU Operator**, the **device
  plugin**, and **GPU Feature Discovery**. The Operator installs `k8s-mig-manager`
  for you — there is no separate Helm chart to install in normal use.
- Root / privileged access: enabling MIG mode and editing partitions requires it,
  and on many GPUs a **GPU reset** (sometimes a node reboot) is needed for the
  mode change to take effect.

> MIG Manager is **a binary + a DaemonSet, not a pip package** — there is nothing
> to `pip install`. The cell below shows the real ways to obtain and enable it.

In [ ]:
# --- Host CLI: get the nvidia-mig-parted binary -----------------------------
# Released from github.com/NVIDIA/mig-parted (pick a tag, e.g. v0.7.0):
# !curl -fsSL -o nvidia-mig-parted \
#   https://github.com/NVIDIA/mig-parted/releases/download/v0.7.0/nvidia-mig-parted-linux-amd64
# !chmod +x nvidia-mig-parted && sudo mv nvidia-mig-parted /usr/local/bin/

# Verify the GPU and driver actually support MIG:
# !nvidia-smi -L                 # lists GPUs; MIG-capable ones say "MIG" once enabled
# !nvidia-smi --query-gpu=mig.mode.current --format=csv

# --- Kubernetes: MIG Manager comes WITH the GPU Operator --------------------
# Enable MIG support when installing the operator (single OR mixed strategy):
# !helm install --wait gpu-operator nvidia/gpu-operator \
#     -n gpu-operator --create-namespace \
#     --set mig.strategy=single
#
# The operator then deploys the nvidia-mig-manager DaemonSet automatically.
# Confirm it is running:
# !kubectl get pods -n gpu-operator -l app=nvidia-mig-manager -o wide
print("nvidia-mig-parted = host CLI; k8s-mig-manager = DaemonSet from the GPU Operator")

## Basic Usage

The mental model: **describe the layout you want in YAML, give it a name, then
`apply` that name.** Below is a minimal config file with a few named layouts for
a single GPU type. `mig-enabled` flips MIG mode; `mig-devices` is a map of
*profile name → count* describing the instances to create.

In [ ]:
# config.yaml -- save this and apply named configs from it
config_yaml = '''
version: v1
mig-configs:
  all-disabled:
    - devices: all
      mig-enabled: false

  all-1g.10gb:                 # 7 small instances per A100-40GB
    - devices: all
      mig-enabled: true
      mig-devices:
        "1g.10gb": 7

  all-balanced:                # mix of sizes on every GPU
    - devices: all
      mig-enabled: true
      mig-devices:
        "1g.10gb": 2
        "2g.20gb": 1
        "3g.40gb": 1

  half-and-half:               # different layout per device index
    - devices: [0, 1, 2, 3]
      mig-enabled: true
      mig-devices: {"3g.40gb": 2}
    - devices: [4, 5, 6, 7]
      mig-enabled: true
      mig-devices: {"1g.10gb": 7}
'''
open("config.yaml", "w").write(config_yaml)
print(config_yaml)

# Host CLI workflow (run as root on the GPU node):
#   nvidia-mig-parted export                       # dump CURRENT layout as YAML
#   nvidia-mig-parted assert -f config.yaml -c all-1g.10gb   # 0 = already in this state
#   sudo nvidia-mig-parted apply  -f config.yaml -c all-1g.10gb
#   nvidia-smi -L                                  # see the 7 MIG devices
#
# Kubernetes workflow -- just set the label; the manager does the rest:
#   kubectl label nodes <node> nvidia.com/mig.config=all-1g.10gb --overwrite

## Advanced Features

#### Hooks: orchestrating the disruptive parts

Toggling MIG mode and (on many GPUs) any partition change requires that **no
process holds the GPU**, and often a **GPU reset**. `nvidia-mig-parted apply
--hooks-file hooks.yaml` runs ordered shell steps at well-defined points so you
can stop clients, reset, and restart them. The k8s-mig-manager ships a hooks
file that stops/starts the operands and resets the GPU; the next cell shows the
shape of one.

#### `assert` for GitOps / CI

`nvidia-mig-parted assert -f config.yaml -c <name>` exits non-zero if the GPU is
**not** already in the named state — perfect for a pre-deploy check or a drift
alarm, without mutating anything.

#### Per-device geometry and valid placements

`mig-devices` counts must form a **valid MIG geometry** — instances consume
fixed slices of compute and memory, and not every combination fits (e.g. you
cannot place two `4g.20gb` on one A100). Use
`nvidia-smi mig -lgip` (list GPU instance profiles) and `-lgipp` (possible
placements) to see what is legal for your GPU before writing a config.

In [ ]:
# hooks.yaml -- steps nvidia-mig-parted runs around an apply (k8s manager uses
# a version of this to stop/start operands and reset the GPU between mode flips)
hooks_yaml = '''
version: v1
hooks:
  # before flipping MIG MODE on/off
  pre-apply-mode:
    - workdir: "/"
      command: "/bin/bash"
      args: ["-c", "systemctl stop nvidia-dcgm.service || true"]

  # before creating/destroying GPU & compute instances
  pre-apply-config:
    - workdir: "/"
      command: "/bin/bash"
      args: ["-c", "echo stopping CUDA clients; fuser -k /dev/nvidia* || true"]

  # after the config has been applied successfully
  post-apply-config:
    - workdir: "/"
      command: "/bin/bash"
      args: ["-c", "systemctl start nvidia-dcgm.service || true"]
'''
open("hooks.yaml", "w").write(hooks_yaml)
print(hooks_yaml)

# Apply WITH hooks (mode change may force a GPU reset; -k keeps going on reset):
#   sudo nvidia-mig-parted apply -f config.yaml -c all-balanced --hooks-file hooks.yaml
#
# Inspect legal instance geometry for THIS GPU before authoring a config:
#   nvidia-smi mig -lgip      # list GPU-instance profiles (sizes + how many fit)
#   nvidia-smi mig -lgipp     # list possible placements
print("\nUse `nvidia-smi mig -lgip` to confirm a layout is physically valid.")

## Use Cases

#### Use Case 1: Multi-tenant inference / notebook serving

- **Context**: A 8×A100 node serves dozens of small models and Jupyter users;
  a full A100 per pod wastes 80%+ of the GPU.
- **Implementation**: Apply `all-1g.10gb` (7 instances/GPU → 56 schedulable MIG
  slices) with the device plugin in `mixed` strategy so pods request
  `nvidia.com/mig-1g.10gb: 1`.
- **Results**: Far higher utilization with **hard isolation** — one tenant's OOM
  or fault cannot crash another's instance.

#### Use Case 2: Right-sized training + inference on the same node

- **Context**: Daytime interactive/inference, nightly fine-tuning.
- **Implementation**: Keep a `mixed` config (`2g.20gb` + a few `1g.10gb`) by day;
  flip nodes to `all-disabled` (full GPUs) at night by relabeling
  `nvidia.com/mig.config`.
- **Results**: One declarative knob retargets the fleet between throughput and
  isolation modes; no manual `nvidia-smi mig` surgery.

#### Use Case 3: CI / ephemeral GPU jobs

- **Context**: Many short test jobs need *a* GPU, not a fast one.
- **Implementation**: Smallest profile (`1g.5gb` on H100, `1g.10gb` on A100) to
  maximize concurrency; `nvidia-mig-parted assert` in CI verifies the node layout
  before scheduling.
- **Results**: High job concurrency and cheap, isolated runners.

## Best Practices

1. **Treat the MIG config as code.** Keep `config.yaml` in Git, review changes,
   and use `nvidia-mig-parted assert` in CI to catch drift before it bites.
2. **Always go through the manager, never raw `nvidia-smi mig`, on managed
   nodes.** Manual changes desync the `nvidia.com/mig.config.state` label and the
   advertised resources from reality.
3. **Drain before reconfiguring.** Partition/mode changes need an idle GPU. On
   k8s the manager drains for you; on bare hosts stop CUDA clients (hooks /
   `fuser -k /dev/nvidia*`) first.
4. **Pick a strategy and stick with it.** `single` (all MIG devices identical,
   advertised as `nvidia.com/gpu`) is simplest; `mixed` (heterogeneous, advertised
   as `nvidia.com/mig-<profile>`) is more flexible but pods must request exact
   profiles.
5. **Plan for the reboot.** On A100 and many cards, enabling MIG mode requires a
   GPU reset or node reboot; size maintenance windows and set
   `WITH_REBOOT=true` on the manager when needed.
6. **Validate geometry up front** with `nvidia-smi mig -lgip/-lgipp`; don't
   hand-guess whether a mix of profiles fits.
7. **Watch `mig.config.state`** and alert on `failed` — a stuck `pending` usually
   means a pod won't drain.

## Common Pitfalls

1. **Editing MIG with `nvidia-smi mig` on a manager-controlled node.** The
   manager's view and the advertised `nvidia.com/mig-*` resources go stale.
   *Avoid*: only change layout via the `nvidia.com/mig.config` label (k8s) or via
   `nvidia-mig-parted apply` on unmanaged hosts.
2. **`state` stuck on `pending`.** Almost always a pod that won't evict (no
   PodDisruptionBudget headroom, local storage, or a non-GPU pod blocking drain).
   *Avoid*: check `kubectl describe node` events and the mig-manager pod logs.
3. **Forgetting the reboot/reset.** Apply "succeeds" but `nvidia-smi` still shows
   MIG disabled. *Avoid*: enable reboot on mode changes; confirm with
   `nvidia-smi --query-gpu=mig.mode.current --format=csv`.
4. **Invalid geometry in the config.** `mig-devices` counts that don't fit a real
   placement make `apply` fail. *Avoid*: validate with `-lgip/-lgipp` first.
5. **Strategy/label mismatch.** Pods request `nvidia.com/mig-1g.10gb` but the
   plugin is in `single` strategy (which advertises `nvidia.com/gpu`), so they
   never schedule. *Avoid*: match pod resource names to the configured strategy.
6. **Config name not in the ConfigMap.** Labeling a node with a name that doesn't
   exist sets state to `failed`. *Avoid*: keep the label values and ConfigMap keys
   in sync.

## Performance Optimization

MIG Manager doesn't add runtime overhead — it *configures* the GPU — so
"performance" here means choosing a partition layout that maximizes useful
throughput for your workload mix.

#### Configuration tuning

- **Profile size vs concurrency**: smaller profiles (`1g.*`) → more instances and
  higher job concurrency, but each instance has fewer SMs and less memory
  bandwidth. Size to the *largest* model/batch a tenant actually needs, not the
  smallest that fits.
- **Memory, not just compute, is the constraint**: a `1g.10gb` slice has a hard
  10 GB cap; an OOM there is isolated but still fatal to that job. Match profile
  memory to model + KV-cache footprint.
- **Don't strand slices**: an A100-40GB yields 7×`1g.10gb` *or* fewer larger ones;
  mixing leaves some compute slices unusable. Use `-lgipp` to avoid layouts that
  waste placements.
- **Bandwidth isolation is the point**: unlike time-slicing/MPS, MIG instances
  don't contend for memory bandwidth — favor MIG when tail latency / noisy-neighbor
  effects matter more than peak single-job speed.

In [ ]:
# Quick "is the layout sane?" check you can run on a node after apply.
# Pretend output parsing -- in practice run the commented nvidia-smi commands.
import subprocess, shutil

def show_mig_layout():
    if not shutil.which("nvidia-smi"):
        print("nvidia-smi not present here; on a GPU node run:")
        print("  nvidia-smi -L")
        print("  nvidia-smi mig -lgi    # list GPU instances")
        print("  nvidia-smi mig -lci    # list compute instances")
        return
    for args in (["nvidia-smi", "-L"],
                 ["nvidia-smi", "mig", "-lgi"]):
        print("$", " ".join(args))
        try:
            print(subprocess.run(args, capture_output=True, text=True, timeout=20).stdout)
        except Exception as e:
            print("  (could not run:", e, ")")

show_mig_layout()

# Confirm MIG mode actually took effect (the #1 missed step):
#   nvidia-smi --query-gpu=index,mig.mode.current --format=csv

## Production Deployment

In production you almost never run the host CLI by hand — the **GPU Operator**
deploys `k8s-mig-manager` and you drive it with **labels** plus a **ConfigMap**.

#### Custom MIG config via ConfigMap

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: custom-mig-parted-config
  namespace: gpu-operator
data:
  config.yaml: |
    version: v1
    mig-configs:
      all-1g.10gb:
        - devices: all
          mig-enabled: true
          mig-devices: {"1g.10gb": 7}
      all-disabled:
        - devices: all
          mig-enabled: false
```

Point the operator at it (via the `ClusterPolicy`), then select per node:

```bash
# Tell the operator which ConfigMap holds the named configs:
kubectl patch clusterpolicy/cluster-policy -n gpu-operator --type merge \
  -p '{"spec":{"migManager":{"config":{"name":"custom-mig-parted-config"}}}}'

# Choose a layout for a node (the manager reconfigures it):
kubectl label node gpu-node-1 nvidia.com/mig.config=all-1g.10gb --overwrite

# Allow the manager to reboot the node when a mode change needs it:
kubectl patch clusterpolicy/cluster-policy -n gpu-operator --type merge \
  -p '{"spec":{"migManager":{"env":[{"name":"WITH_REBOOT","value":"true"}]}}}'
```

A workload then requests MIG slices (mixed strategy):

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: mig-inference
spec:
  containers:
    - name: app
      image: nvcr.io/nvidia/pytorch:24.05-py3
      resources:
        limits:
          nvidia.com/mig-1g.10gb: 1     # one isolated MIG instance
```

## Monitoring and Observability

#### Key signals to track

- **`nvidia.com/mig.config.state`** on each node — the rollout status
  (`pending` / `success` / `failed`). Alert on `failed` and on `pending` that
  lasts longer than your drain timeout.
- **`nvidia.com/mig.config`** vs actual layout — drift between the requested label
  and `nvidia-smi -L` / `nvidia-mig-parted export` output.
- **Per-instance utilization** from **DCGM / dcgm-exporter** — DCGM reports
  metrics per MIG instance (e.g. `DCGM_FI_PROF_GR_ENGINE_ACTIVE`), so you can see
  whether slices are actually used or stranded.
- **Schedulable MIG resources** — `kubectl describe node` should show
  `nvidia.com/mig-<profile>` (mixed) or an updated `nvidia.com/gpu` (single)
  capacity after a reconfigure.

```bash
# Rollout state across the fleet:
kubectl get nodes -L nvidia.com/mig.config -L nvidia.com/mig.config.state

# What the manager actually did (and why a reconfigure stalled):
kubectl logs -n gpu-operator -l app=nvidia-mig-manager --tail=200

# Advertised MIG resources on a node:
kubectl describe node gpu-node-1 | grep -i nvidia.com/mig
```

#### Logging best practices

- Keep the mig-manager pod logs in your aggregation pipeline — they record drain,
  apply, reset, and restart steps in order, which is exactly what you need when a
  reconfigure goes sideways.
- Correlate a `failed` state with the node's drain/eviction events.

## Troubleshooting

#### Issue 1: `nvidia.com/mig.config.state` is stuck on `pending`

**Symptoms**: A relabeled node never reaches `success`; new MIG resources don't
appear.

**Cause**: The manager cordoned the node but a GPU pod won't evict (PDB blocks it,
uses local storage, or is not managed by a controller). The apply can't start
until the GPU is idle.

**Solution**: `kubectl get pods -A --field-selector spec.nodeName=<node>` and the
mig-manager logs to find the blocker; fix the PDB / delete the stuck pod, or set
the manager's drain options appropriately.

#### Issue 2: Apply reports success but MIG is still disabled

**Symptoms**: `mig.config.state=success` yet `nvidia-smi` shows MIG mode off and
no instances.

**Cause**: Enabling MIG mode needed a GPU reset/reboot that didn't happen.

**Solution**: Set `WITH_REBOOT=true` on the migManager so it reboots when a mode
change requires it; verify with
`nvidia-smi --query-gpu=mig.mode.current --format=csv`.

#### Issue 3: State goes to `failed`

**Symptoms**: Node shows `mig.config.state=failed` right after relabeling.

**Cause**: The config name isn't in the ConfigMap, or the requested geometry is
invalid for that GPU.

**Solution**: Confirm the label value matches a key under `mig-configs:` in the
ConfigMap; validate geometry with `nvidia-smi mig -lgip/-lgipp`; check
mig-manager logs for the exact `nvidia-mig-parted` error.

## Comparison with Alternatives

How MIG (configured by MIG Manager) compares with the other GPU-sharing options:

| Aspect | MIG + MIG Manager | Time-Slicing | MPS (Multi-Process Service) |
|---|---|---|---|
| Isolation | **Hard** — separate SMs, L2, memory & bandwidth; fault-isolated | None — processes time-share, no memory isolation | Soft — shared context, optional memory limits |
| Hardware | A100/A30/H100/H200/B200 only | Any NVIDIA GPU | Any NVIDIA GPU (Volta+) |
| Reconfigure cost | Disruptive: drain + possible GPU reset/reboot | Free, instant (just replicas) | Cheap, per-node daemon |
| Granularity | Fixed profiles (1g/2g/3g/4g/7g) | Arbitrary replica count (oversubscribe) | Arbitrary processes, optional % cap |
| Best for | Multi-tenant isolation, predictable QoS | Bursty/low-utilization sharing, dev | Many cooperative processes, one trusted tenant |

### When to choose MIG Manager

Choose MIG (and MIG Manager to configure it) when:

- You need **hard isolation / QoS guarantees** between tenants on one GPU.
- You have **MIG-capable hardware** and many small-to-medium jobs.
- You want partition layout to be **declarative and fleet-managed**, not manual.

Prefer **time-slicing** when you just want cheap oversubscription on any GPU and
don't need isolation; prefer **MPS** when one trusted workload has many
cooperating processes. These are complementary — you can even time-slice *within*
a MIG instance.

## Resources

### Official documentation

- MIG Manager / mig-parted source: <https://github.com/NVIDIA/mig-parted>
- GPU Operator — MIG support docs: <https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-operator-mig.html>
- NVIDIA MIG User Guide: <https://docs.nvidia.com/datacenter/tesla/mig-user-guide/>
- k8s-device-plugin MIG strategies: <https://github.com/NVIDIA/k8s-device-plugin#configuration-option-details>

### Tutorials and guides

- "Getting the Most Out of the A100 GPU with MIG" (NVIDIA technical blog):
  <https://developer.nvidia.com/blog/getting-the-most-out-of-the-a100-gpu-with-multi-instance-gpu/>
- GPU Operator MIG how-to (label-driven reconfiguration):
  <https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-operator-mig.html#configuring-mig-profiles>
- `nvidia-mig-parted` README & examples: <https://github.com/NVIDIA/mig-parted/tree/main/examples>

### Community resources

- NVIDIA Developer Forums (GPU virtualization / MIG):
  <https://forums.developer.nvidia.com/>
- GPU Operator issues / discussions:
  <https://github.com/NVIDIA/gpu-operator/issues>
- Stack Overflow tag: <https://stackoverflow.com/questions/tagged/nvidia-mig>

### Related technologies

- **Multi-Instance GPU (MIG)** — the hardware feature MIG Manager configures.
- **NVIDIA GPU Operator** — installs and drives the k8s-mig-manager.
- **k8s-device-plugin & GPU Feature Discovery** — advertise MIG resources/labels.
- **DCGM / dcgm-exporter** — per-MIG-instance metrics.
- **Time-Slicing & MPS** — the alternative GPU-sharing mechanisms.